# 12B · The Fragility Lab — When the Optimiser Meets Reality
### Financial Analytics — Module 12 · Lab 4

Module 7 left a warning ticking: *"optimisers are drama queens about inputs — remember this in Module 12."* Time's up. This notebook stress-tests everything 12A built:

1. **The drama-queen demo:** nudge one expected return, watch the allocation convulse
2. **The estimation-error reality:** how wrong ARE return estimates? (Very. Measurably.)
3. **The out-of-sample tournament:** optimised portfolios vs naive **1/N** — the famous humbling
4. **What survives:** the honest, defensible way professionals actually use this machinery

> 🛡️ **Bias check:** the tournament below is the look-ahead test IN REVERSE — we deliberately estimate on one period and deploy on another, exactly as real life forces. Survivor universe (stated, flatters ALL contestants equally). Regime: the split point (end-2023) also splits regimes — feature, not bug: real deployment always crosses regimes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
sns.set_theme(style="whitegrid")

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])
px = uni.pivot(index="date", columns="ticker", values="close").sort_index()
px.loc[:"2024-09-01", "TATAMOTORS.NS"] = px.loc[:"2024-09-01", "TATAMOTORS.NS"] / 5
rets = px.pct_change().dropna()
n = rets.shape[1]

def max_sharpe_weights(mu_v, cov_m, cap=1.0):
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    res = minimize(lambda w: -(w @ mu_v)/np.sqrt(w @ cov_m @ w), np.ones(n)/n,
                   bounds=[(0, cap)]*n, constraints=cons, method="SLSQP")
    return res.x

---
## 1. The drama queen, caught on camera

Estimate inputs from the full sample. Then nudge ONE stock's expected return by ±2 percentage points — well inside any honest confidence interval — and re-optimise:

In [ ]:
mu  = (rets.mean()*252).values
cov = (rets.cov()*252).values
tickers = [t.replace(".NS","") for t in px.columns]

w_base = max_sharpe_weights(mu, cov)
target_idx = int(np.argmax(w_base))          # the optimiser's current favourite
print(f"Optimiser's favourite: {tickers[target_idx]} at {w_base[target_idx]:.0%}\n")

rows = {}
for bump in [-0.02, 0.0, +0.02]:
    mu_b = mu.copy(); mu_b[target_idx] += bump
    w = max_sharpe_weights(mu_b, cov)
    rows[f"{bump:+.0%} bump"] = w
W = pd.DataFrame(rows, index=tickers)
print((W[(W > 0.005).any(axis=1)]*100).round(1).to_string())
turnover = np.abs(W.iloc[:,2] - W.iloc[:,0]).sum()/2
print(f"\nA 2pp nudge - noise-sized - reshuffles {turnover:.0%} of the portfolio. THE DRAMA QUEEN, confirmed.")

**Why so hysterical?** Because to an optimiser, the stock with the best estimated return at similar risk isn't "slightly preferable" — it's *strictly better*, so pile in until a constraint objects. Module 7's warning verbatim: **an error and an opportunity look identical to a solver.** And return estimates are nothing BUT error, as we now measure:

## 2. How wrong are the inputs, really?

In [ ]:
# Estimate mu and vol on 2022-23; compare against what 2024-25 actually delivered
split = "2023-12-31"
r1, r2 = rets[:split], rets[split:]
est = pd.DataFrame({
    "mu_est":  r1.mean()*252, "mu_real": r2.mean()*252,
    "vol_est": r1.std()*np.sqrt(252), "vol_real": r2.std()*np.sqrt(252)})

print(f"corr(estimated mu, realised mu):   {est['mu_est'].corr(est['mu_real']):+.2f}   <- returns: estimates near-USELESS")
print(f"corr(estimated vol, realised vol): {est['vol_est'].corr(est['vol_real']):+.2f}   <- risk: estimates genuinely informative")
print()
print("The asymmetry that runs this whole field: RISK is forecastable (9B: volatility has memory);")
print("RETURN is not (6B: prices eat their own forecasts). Yet max-Sharpe leans HARDEST on the")
print("un-forecastable input. That mismatch is the fragility - and the tournament will price it.")

---
## 3. The tournament: optimised vs 1/N

Rules of honest engagement: **estimate on 2022–23, deploy untouched on 2024–25.** Four contestants:
- **Max-Sharpe** (the star of 12A, unconstrained)
- **Max-Sharpe, 10% capped** (constraints as humility)
- **Min-variance** (uses only the covariance — the forecastable input)
- **1/N equal weight** (estimates NOTHING — the humble control)

In [ ]:
mu1, cov1 = (r1.mean()*252).values, (r1.cov()*252).values

def min_var_weights(cov_m, cap=1.0):
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    res = minimize(lambda w: w @ cov_m @ w, np.ones(n)/n, bounds=[(0,cap)]*n, constraints=cons, method="SLSQP")
    return res.x

contestants = {
    "max-Sharpe":        max_sharpe_weights(mu1, cov1),
    "max-Sharpe cap10%": max_sharpe_weights(mu1, cov1, cap=0.10),
    "min-variance":      min_var_weights(cov1),
    "1/N equal":         np.ones(n)/n,
}

print(f"{'portfolio':<20}{'IN-SAMPLE Sharpe':>18}{'OUT-OF-SAMPLE Sharpe':>22}{'OOS vol':>10}{'OOS maxDD':>11}")
for name, w in contestants.items():
    def sh(r): 
        pr = r @ w
        return (pr.mean()*252)/(pr.std()*np.sqrt(252))
    pr2 = r2 @ w
    eq = (1+pr2).cumprod()
    dd = (eq/eq.cummax()-1).min()
    print(f"{name:<20}{sh(r1):>18.2f}{sh(r2):>22.2f}{pr2.std()*np.sqrt(252):>10.1%}{dd:>11.1%}")

**Read the two Sharpe columns slowly — this table is the lab.**

- **In-sample**, max-Sharpe crushes everyone. Of course it does: it was *fitted* to that period. In-sample Sharpe is the equity-curve version of a self-graded exam.
- **Out-of-sample**, the ranking scrambles. The unconstrained star typically falls to earth (its concentrated bets were bets on estimation noise); the capped version holds up better; **min-variance** — which never touched the toxic input — is sturdy; and **1/N**, which estimated nothing, competes embarrassingly well.

This is not our data being quirky. It is one of the most replicated findings in modern finance (the famous "1/N" studies): **naive diversification beats optimised portfolios out-of-sample with humiliating regularity**, because optimisation's edge is smaller than estimation error's damage. Markowitz won the Nobel for the frontier — and reportedly held his own retirement in something close to 1/N.

## 4. So is the frontier useless? No — here's what survives

In [ ]:
# What professionals keep from this wreckage:
print("""THE SURVIVORS' LIST
1. DIVERSIFICATION itself - the correlation bend is real physics; 12A Ex1 measured it. Never in doubt.
2. THE COVARIANCE MATRIX - risk structure is forecastable; min-variance and risk-budgeting
   use only what can be estimated. That's why 'low-vol' and 'risk-parity' products exist at scale.
3. CONSTRAINTS AS HUMILITY - caps, sector limits, turnover limits. They 'cost' in-sample Sharpe
   and BUY out-of-sample survival (the cap-10% row). Module 7's boardroom rules, vindicated.
4. THE FRONTIER AS A MAP, not a GPS - it teaches the SHAPE of the trade-off and what's impossible.
   Professionals use it to frame choices; only the reckless use raw max-Sharpe weights.
5. BETTER INPUTS BEAT BETTER OPTIMISERS - shrinking estimates toward priors (awareness-level:
   'shrinkage', 'Black-Litterman') attacks the disease, not the symptom. The next course's door.
""")

### ✏️ Exercises
1. **The turnover tax:** re-run the tournament assuming each portfolio is re-optimised quarterly in 2024–25, paying 15 bps on turnover (reuse 11B's cost logic). Which contestant suffers most, and why is the answer obvious in hindsight?
2. **Shrinkage, homemade:** blend the estimates — `mu_shrunk = 0.3*mu1 + 0.7*mu1.mean()` (pull every stock toward the grand average). Re-run max-Sharpe OOS. Better or worse than raw? You've just reinvented the first 30% of a famous technique.
3. **The client letter:** in five sentences, explain to a client why you hold the capped portfolio instead of the "optimal" one, without the words 'covariance', 'Sharpe', or 'estimation error'. (This is 12.5's skill, previewed — and harder than the optimisation.)

---
## Lab 4 complete — and Part D with it

Correlation bends risk · the frontier is the cloud's edge · optimisers amplify their worst input · risk is forecastable, return isn't · 1/N is the benchmark that keeps everyone honest · constraints are purchased survival. Four labs: series read, futures simulated, strategies audited, portfolios built and humbled. **Badge: Frontier Walker 🏔️**

*AI disclosure: ______*

In [ ]:
# workspace
